In [7]:
display(config_box)
display(launch_button)
display(output_area)
# Bienvenue cliquez ICI en 2 après !!!! :) 

Button(button_style='success', description='▶ LANCER SIMULATION', icon='play', style=ButtonStyle())

Output(outputs=({'output_type': 'stream', 'text': '\n=========================================================…

In [1]:

# 1er clique
#--------------------------
# 2026 06 13 version v9.12 ************************************
# *
# *   ███████╗██████╗ ██╗  ██╗██╗██████╗ ██╗████████╗
# *   ██╔════╝██╔══██╗██║  ██║██║██╔══██╗██║╚══██╔══╝
# *   ███████╗██████╔╝███████║██║██████╔╝██║   ██║
# *   ╚════██║██╔═══╝ ██╔══██║██║██╔══██╗██║   ██║
# *   ███████║██║     ██║  ██║██║██║  ██║██║   ██║
# *   ╚══════╝╚═╝     ╚═╝  ╚═╝╚═╝╚═╝  ╚═╝╚═╝   ╚═╝
# *
# *   Smooth Particle Hydrodynamics
# *   Integrated Resource for Instruction and Teaching
# *
# *   Educational & Research CFD Platform
# *
# ****************************************************************/
#****************************************************************
# Masse conservation (no variation) and no volume variation
# ALE formulation
# Ordre 2 and 3 in time and space with MLS and Renormalization for div operator 
# Six tests cas  : 
#                   jet impact, impact of a column of water on a plate or deux jets impacted
#                   funnel , vsicosity study via water release Funnel Discharge Benchmark (FDB)
#                   DamBreak, ALE, L dambreak
#                   TGV, : ALE,El L vortex preservation
#                   car (FSI)  : full lagrangian interaction between a car (VBL) and a water tank
#                   floating body  https://www.spheric-sph.org/tests/test-12
#                   Test mesh mouvement (no Euler or Navier-Stokes resolution only mesh displacement) : based on TGV
#                   Poiseuille
# ---- TO-DO--------------
#  OPtimisation des calculs (vitesse, doublon etc)
#  protocole funnel faire avec Baptiste
#  faire Poiseuille convergence log-log
#  faire convergence TGV eulerian, lagrangian log-log
#  shifting ++
#  theorical aspect
# --------------------------  
# Structure du code
#1. Paramètres physiques et numériques per testcase
#2. Fonctions SPH (kernel, gradient, pression, densité, Riemann)
#3. Fonctions ALE (entropie, v_mesh)
#4. Initialisation maillage + particules
#5. Boucle temporelle :
  # a) Mise à jour conditions aux frontières fantômes
 #  b) Calculs Shepard ou RENORM / ALE / v_mesh
 #  c) Calculs flux massiques et accélérations
 #  d) Mise à jour rho, vel, pos
 #  e) Calcul dt CFL
  # f) Sauvegarde VTK
# ------------------------
# Biblio 
#test case TGV and jet2D in https://www.sciencedirect.com/science/article/pii/S0045782523002839
#test case DamBreak  https://www.spheric-sph.org/tests/test-02 et https://www.tandfonline.com/doi/full/10.1080/21664250.2019.1672124?utm_source=chatgpt.com#d1e3653
#test case funnel https://cdn.standards.iteh.ai/samples/73851/ccc06afc20a046ac83a46e30264f221c/ISO-2431-2019.pdf
#test case car
#test case Poiseuille 
from IPython.display import HTML, Javascript, display as ipy_display
import numpy as np
import os
from scipy.spatial import cKDTree
import shutil
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import ipywidgets as widgets
from IPython.display import display, HTML
from scipy.spatial import Voronoi
# Cellule 1 : Widgets uniquement
import ipywidgets as widgets
from IPython.display import display, HTML

# ============ INITIALISER ============
testCase = 'DamBreak'
mesh_mode = 'lagrangian'
mode_ale = 'wass'
beta_geom = 0.001
alpha = 0.01
mode_p_refinement = 0
mode_RK = 2
Kernel = True
Relax = True
RenormON = False
ShepardCorr = False
ViscoONOFF = False
ShiftPart = False
ShiftMethod = 'delta_sph'  # 'delta_sph' ou 'lloyd'
ShiftCFL = 0.05            # Coefficient stabilité (0.01-0.1)
ShiftR = 0.2               # Ratio remplissage (0.1-0.3)
ShiftBetaLloyd = 0.5       # Coefficient Lloyd
Filtre = False
randompos = True

ModeDEBUG = False
ModeGEOM = False
CFL = 0.1
dx = 1
h = 2.0
C0 = 7.0


Button(button_style='success', description='▶ LANCER SIMULATION', icon='play', style=ButtonStyle())

Output()